# Porownanie architektur cz.2 — domkniecie kwadratu CNN/Transformer x maly/duzy

Teza (z notes/07): transformery overfituja do synthetic i gorzej transferuja niz CNN.
Dotad: YOLOv10n (CNN maly) 0.459 vs RT-DETR-l (Transf. maly) 0.297 na realnym tescie.

Tu domykamy druga os — DUZE modele:
- **RT-DETR-x** (~67M, transformer duzy) — czy wiekszy transformer transferuje JESZCZE gorzej?
- **YOLO11l** (~25M, CNN duzy) — czy nowszy/wiekszy CNN trzyma poziom transferu?

Trening na synthetic 10k, ewaluacja cross-domain na realnym holdoucie. A100, ~10h lacznie (2 modele).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!git clone https://github.com/JakubPaszke/rareplanes-domain-shift.git
%cd rareplanes-domain-shift
!pip -q install ultralytics pycocotools

In [ ]:
# Dane: adnotacje + realny test + synthetic 10k (Python downloader)
import os, urllib.request
from concurrent.futures import ThreadPoolExecutor
for d in ['data/real/annotations','data/real/tarballs','data/synthetic/annotations','data/synthetic/images/train']:
    os.makedirs(d, exist_ok=True)
B='https://rareplanes-public.s3.amazonaws.com'
for f in ['instances_train_aircraft.json','instances_test_aircraft.json']:
    !curl -sf -o data/real/annotations/{f} {B}/real/metadata_annotations/{f}
    !curl -sf -o data/synthetic/annotations/{f} {B}/synthetic/metadata_annotations/{f}
!curl -sf -o data/real/tarballs/test.tar.gz {B}/real/tarballs/test/RarePlanes_test_PS-RGB_tiled.tar.gz
!mkdir -p data/real/PS-RGB_tiled && tar -xzf data/real/tarballs/test.tar.gz -C data/real/PS-RGB_tiled/
DST='data/synthetic/images/train'
files=[l.strip() for l in open('configs/synthetic_10k_train_files.txt')]+[l.strip() for l in open('configs/synthetic_10k_val_files.txt')]
files=[f for f in files if f]
def fetch(fn):
    dst=f'{DST}/{fn}'
    if os.path.exists(dst) and os.path.getsize(dst)>0: return True
    try:
        urllib.request.urlretrieve(f'{B}/synthetic/train/images/{fn}', dst+'.part'); os.rename(dst+'.part',dst); return True
    except Exception:
        if os.path.exists(dst+'.part'): os.remove(dst+'.part')
        return False
with ThreadPoolExecutor(max_workers=32) as ex: ok=sum(ex.map(fetch, files))
have=len([f for f in os.listdir(DST) if f.endswith('.png')])
print(f'pobrano {ok}/{len(files)}, na dysku {have}'); assert have>len(files)*0.99
!python3 src/coco_to_yolo.py --domain synthetic --classes aircraft --val-frac 0.15

In [ ]:
# A) RT-DETR-x (transformer duzy) — recepta stabilna lr=1e-4
!python3 src/train_yolo.py --data data/yolo/synthetic_aircraft/data.yaml \
  --name rtdetrx_10k --model rtdetr-x.pt \
  --epochs 60 --batch 8 --imgsz 512 --seed 42 --patience 20 --workers 8 \
  --optimizer AdamW --lr0 0.0001 --cos_lr --warmup_epochs 5
!python3 src/eval_per_size.py --weights runs/rtdetrx_10k/weights/best.pt \
  --img-dir data/real/PS-RGB_tiled/PS-RGB_tiled \
  --coco-gt data/real/annotations/instances_test_aircraft.json --name rtdetrx_10k_ml

In [ ]:
# B) YOLO11l (CNN duzy) — domyslny optimizer/lr (YOLO 'po prostu dziala')
!python3 src/train_yolo.py --data data/yolo/synthetic_aircraft/data.yaml \
  --name yolo11l_10k --model yolo11l.pt \
  --epochs 60 --batch 16 --imgsz 512 --seed 42 --patience 20 --workers 8
!python3 src/eval_per_size.py --weights runs/yolo11l_10k/weights/best.pt \
  --img-dir data/real/PS-RGB_tiled/PS-RGB_tiled \
  --coco-gt data/real/annotations/instances_test_aircraft.json --name yolo11l_10k_ml

In [ ]:
# Podsumowanie + AUTO-DOWNLOAD obu wynikow
import json
for n in ['rtdetrx_10k_ml','yolo11l_10k_ml']:
    m=json.load(open(f'results/per_size/{n}.json'))['metrics']
    print(f"{n:18s} mAP50={m['AP@.5']:.3f} S={m['AP_small']:.3f} M={m['AP_medium']:.3f} L={m['AP_large']:.3f}")
from google.colab import files
files.download('results/per_size/rtdetrx_10k_ml.json')
files.download('results/per_size/yolo11l_10k_ml.json')

## Oczekiwany kwadrat (real test mAP@50)
| | maly | duzy |
|---|---|---|
| CNN | YOLOv10n 0.459 | YOLO11l ? |
| Transformer | RT-DETR-l 0.297 | RT-DETR-x ? |

Hipoteza: RT-DETR-x <= 0.297 (wiekszy transformer = wiekszy overfit), YOLO11l ~0.45+ (CNN trzyma transfer).
Jesli tak — teza 'transformery gorzej transferuja, pojemnosc szkodzi' mocno potwierdzona.

Uwaga: RT-DETR-x batch=8 (67M param, wieksze VRAM); jesli OOM na A100-40GB, zmniejsz do batch=4.